<h1>Ensemble - Bagging</h1>

Ensemble bedeutet, dass mehrere Algorithmen f?r eine Antwort verwendet werden, z. B. als Mehrheitsentscheidung wie bei Random-Forest.

Bei Random-Forest wird der Algorithmus f?r Entscheidungsb?ume genommen, die mehrere B?ume erzeugen und mehrere Antworten liefern. Durch eine Mehrheitsentscheidung gibt es am Ende nur eine Antwort, genau das ist Esemble Learning.

Ensemble Learning bietet zwei Methoden an-
- Bagging (Bootstrap Aggregation)
- Boosting

Diese k?nnen bei Modellen eingesetzt werden, wenn Datasets durch deren Struktur, Eigenschaften, Aufteilung, ..., daf?r sorgen, das das Model beim Trainieren gut punktet, aber beim Testen eine hohe Varianz aufweist. <br>
Hohe Varianz eines Models (Test errors) => Indikator f?r Overfitting. 

<i>Abb1</i>: Bagging - Resampling with Replacement. 

<img src="./files_data/img/sklearn_ensemble_1.PNG" width=650 hight=450> 

Aus einem gegebenen Dataset werden n-Samples zuf?llig gezogen. Die Wahrscheinlichkeit der Ziehung ist gleich verteilt, und es k?nnen Duplikate vorkommen. Dieser Vorgang mit Resampling with replacement nennt sich Bootstrap. 

Es k?nnen n-Subsets eines Datasets erstellt werden, die dann je ein Model bedienen. Genau so funktioniert auch Random-Forest.
Jeder dieser Modelle hat unterschiedliche Subsets gef?llt mit anderen Daten, was auch zu anderen Antworten f?hrt. Durch eine Mehrheitsentscheidung wird festgestellt, welche Antwort genommen wird => z. B. True / False

Jeder der Sub-Modelle f?hren eine Prediction aus. Jeder dieser Sub-Modelle wird als weak learner bezeichnet (weil mit Subset trainiert).

<i>Abb2</i>: Mehrheitsentscheidung bei Bagging, Gesamt?berblick. 

<img src="./files_data/img/sklearn_ensemble_2.PNG" width=650 hight=650> 

Die kombinierten Ergebnisse der einzelnen Sub-Modelle liefern allgemein gute Ergebnisse.
- Bei Regression wird der Durchschnitt der Antworten genommen.

Random-Forest an sich ist ein Model, das schon Bagging nutzt und auch die Features sampelt. Jedes Subset bildet einen Baum. <br>
Ein einziger Baum, oder ein einzelnes Model kann zu Overfitting f?hren => keine gute Balance. <br>
Mehrere Modelle k?nnen das Dataset aus verschiedenen Winkeln betrachten und zusammen ein besseres Ergebnis liefern.

F?r die Anwendung von Bagging nutzen wir ein Heart-Failure Dataset und SVM, dann wird der Score mit und ohne Bagging verglichen.

In [ ]:
# Imports
import pandas as pd   

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm  import SVC

from sklearn.ensemble import BaggingClassifier

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing   import StandardScaler  
from sklearn.preprocessing   import LabelEncoder

In [ ]:
# Lade das Dataset in ein Dataframe.
heart_data = pd.read_csv("./files_data/data/heart_failure_prediction.zip", compression='zip')
heart_data.head(2)

In [ ]:
# Gibt es Null-Werte?
heart_data.isnull().sum()

Ohne weiter ins Detail zu gehen, werden die Features encoded und skaliert.  <br>

In [ ]:
# On Hot Encode mit Pandas
dummies_df = pd.get_dummies(heart_data[['Sex', 'ExerciseAngina']], dtype="int")
# L?sche betroffene Spalten.
heart_data.drop(['Sex', 'ExerciseAngina'], axis='columns', inplace=True)
# H?nge neue Spalten an.
heart_data = pd.concat([dummies_df, heart_data ], axis='columns')

# Erstelle Sklearn Label Encoder.
le = LabelEncoder()
# Encode Features.
heart_data['ChestPainType']  = le.fit_transform(heart_data['ChestPainType'])
heart_data['RestingECG']     = le.fit_transform(heart_data['RestingECG'])
heart_data['ST_Slope']       = le.fit_transform(heart_data['ST_Slope'])

heart_data.head(5)

Optional k?nnen die Daten skaliert werden. 

In [ ]:
scaler = StandardScaler()
X_data = scaler.fit_transform(heart_data.drop(['HeartDisease'], axis="columns"))
X_data.shape

In [ ]:
# Erstelle Model.
svc = SVC()

cross_val_score(svc, X_data, heart_data['HeartDisease'], cv=4)

Mit Cross Validation 4 ist zu sehen, dass der Score sich je nach Aufteilung des Datasets im Score unterscheidet.
- Model hat eine h?here Varianz

Was machen k?nnte, ist Bagging zu verwenden (was auch den Anwendungszeck zeigen soll, deswegen haben wir das Model nicht optimiert...).

In [ ]:
# Dasselbe kann auch mit einem Baum gemacht werden. 
# - Auch eine h?here Varianz. 
dtree = DecisionTreeClassifier()

cross_val_score(dtree, X_data, heart_data['HeartDisease'], cv=4)

In [ ]:
# Einige Parameter, die eingestellt werden k?nnen
# - Siehe Sklearn Doks. 
model = BaggingClassifier( 
    estimator= svc,        # Welches Model. 
    n_estimators  = 10,    # Wie viele Modelle => n-Subsets -> n-Modelle
    max_samples   = 0.85,  # Pool f?r Subsets => 100 Samples und max_samples=0.85 -> Subset hat 85 Samples. 
    oob_score     = True   # Out of Bag. Wenn Datenpunk nicht gezogen wird => Nutze es als Testset. Nur bei Bootstrap. 
    # verbose       = 2    # Zus?tzliche Informationen => kann viel Text generieren
)
model

In [ ]:
# Trainiere.
model.fit(X_data, heart_data['HeartDisease'])
model.oob_score_

In [ ]:
# SVM
model = BaggingClassifier( 
    estimator= svc,        # Welches Model. 
    n_estimators  = 20,    # Wie viele Modelle => n-Subsets -> n-Modelle
    max_samples   = 0.85,  # Pool f?r Subsets => 100 Samples und max_samples=0.85 -> Subset hat 85 Samples. 
    #oob_score     = True  # Out of Bag. Wenn Datenpunk nicht gezogen wird => Nutze es als Testset. Nur bei Bootstrap. 
    # verbose       = 2    # Zus?tzliche Informationen => kann viel Text generieren
)

cross_val_score(model, X_data, heart_data['HeartDisease'], cv=4)

In [ ]:
# Baum 
model = BaggingClassifier( 
    estimator= dtree,      # Welches Model. 
    n_estimators  = 20,    # Wie viele Modelle => n-Subsets -> n-Modelle
    max_samples   = 0.85,  # Pool f?r Subsets => 100 Samples und max_samples=0.85 -> Subset hat 85 Samples. 
    #oob_score     = True  # Out of Bag. Wenn Datenpunk nicht gezogen wird => Nutze es als Testset. Nur bei Bootstrap. 
    # verbose       = 2    # Zus?tzliche Informationen => kann viel Text generieren
)

cross_val_score(model, X_data, heart_data['HeartDisease'], cv=4)

Zusammengefasst:

In [ ]:
# SVM Score # 
svc = SVC()
cross_val_score(svc, X_data, heart_data['HeartDisease'], cv=4).mean()

In [ ]:
# SVM Score mit Bagging # 
model = BaggingClassifier( 
    estimator= svc,       
    n_estimators  = 10,   
    max_samples   = 0.78,  
)
cross_val_score(model, X_data, heart_data['HeartDisease'], cv=4).mean()

In [ ]:
# Baum #
dtree = DecisionTreeClassifier()
cross_val_score(dtree, X_data, heart_data['HeartDisease'], cv=4).mean()

In [ ]:
# Baum mit Bagging # 
model = BaggingClassifier( 
    estimator= dtree,      
    n_estimators  = 20,    
    max_samples   = 0.85,
)

cross_val_score(model, X_data, heart_data['HeartDisease'], cv=4).mean()

In [ ]:
# Baum mit Bagging und Feature sampling (wie Random-Forest) # 
model = BaggingClassifier( 
    estimator= dtree,      
    n_estimators  = 20,    
    max_samples   = 0.85,   
    max_features  = 0.90  # Feature pool. 
)

cross_val_score(model, X_data, heart_data['HeartDisease'], cv=4).mean()

In [ ]:
# // Content Coming [Boosting, Stacking, Voting]